# Autoresearch Experiment Analysis

Analysis of autonomous research results from `results.jsonl`.

Keyed off `score` (whatever `objective.GOAL` was set to) rather than `val_bpb`
directly, so this notebook keeps working when the goal changes. Falls back to the
legacy `results.tsv` schema if no JSONL log exists yet.

In [ ]:
import os
import sys

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

sys.path.insert(0, os.getcwd())
import objective

rows = objective.load_results()
if rows:
    df = pd.DataFrame(rows)
    print(f"Loaded {len(df)} runs from {objective.RESULTS_PATH}")
elif os.path.exists("results.tsv"):
    # Legacy 5-column schema, so runs from before the JSONL switch still analyse
    df = pd.read_csv("results.tsv", sep="\t").rename(
        columns={"description": "note", "memory_gb": "peak_vram_gb"})
    df["score"] = df["val_bpb"]
    df["goal"] = "min_bpb"
    print(f"Loaded {len(df)} runs from legacy results.tsv")
else:
    raise SystemExit(
        f"No results yet.\n"
        f"  looked for: {objective.RESULTS_PATH}\n"
        f"  and:        {os.path.abspath('results.tsv')}\n"
        f"Run an experiment first (`uv run train.py`), or start this notebook "
        f"from the repo root.")

df["status"] = df["status"].astype(str).str.strip().str.lower()
for col in ("score", "val_bpb", "peak_vram_gb", "num_params_M", "training_seconds"):
    if col in df:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# Scores from different goals are not comparable — analyse one at a time.
if "goal" in df and df["goal"].nunique() > 1:
    latest = df["goal"].dropna().iloc[-1]
    print(f"WARNING: log contains mixed goals {sorted(df['goal'].dropna().unique())}; "
          f"filtering to {latest!r}")
    df = df[df["goal"] == latest].reset_index(drop=True)

print(f"Goal:    {df['goal'].iloc[0] if 'goal' in df and len(df) else 'unknown'}")
print(f"Columns: {list(df.columns)}")
df.head(10)

In [ ]:
counts = df["status"].value_counts()
print("Experiment outcomes:")
print(counts.to_string())

n_keep = counts.get("keep", 0)
n_discard = counts.get("discard", 0)
n_crash = counts.get("crash", 0)
n_pending = counts.get("pending", 0)
n_decided = n_keep + n_discard
if n_decided > 0:
    print(f"\nKeep rate: {n_keep}/{n_decided} = {n_keep / n_decided:.1%}")
if n_pending:
    print(f"{n_pending} run(s) still pending a keep/discard decision")

In [ ]:
# Show all KEPT experiments (the improvements that stuck)
kept = df[df["status"] == "keep"].copy()
print(f"KEPT experiments ({len(kept)} total):\n")
for i, row in kept.iterrows():
    vram = row.get("peak_vram_gb")
    vram_s = f"mem={vram:.1f}GB  " if pd.notna(vram) else ""
    print(f"  #{i:3d}  score={row['score']:.6f}  {vram_s}{row.get('note', '')}")

## Val BPB Over Time

Track how the best (kept) val_bpb evolves as experiments progress. The running minimum shows the "frontier" -- the best result achieved so far.

In [ ]:
fig, ax = plt.subplots(figsize=(16, 8))
goal = df["goal"].iloc[0] if "goal" in df and len(df) else "unknown"

# Crashes and undecided runs have no score to plot
valid = df[(df["status"] != "crash") & df["score"].notna()].copy()
valid = valid[np.isfinite(valid["score"])].reset_index(drop=True)

baseline = valid.loc[0, "score"]

# Only plot points at or below baseline (the interesting region)
below = valid[valid["score"] <= baseline + 0.0005]

disc = below[below["status"] == "discard"]
ax.scatter(disc.index, disc["score"],
           c="#cccccc", s=12, alpha=0.5, zorder=2, label="Discarded")

kept_v = below[below["status"] == "keep"]
ax.scatter(kept_v.index, kept_v["score"],
           c="#2ecc71", s=50, zorder=4, label="Kept", edgecolors="black", linewidths=0.5)

kept_mask = valid["status"] == "keep"
kept_idx = valid.index[kept_mask]
kept_score = valid.loc[kept_mask, "score"]
running_min = kept_score.cummin()
ax.step(kept_idx, running_min, where="post", color="#27ae60",
        linewidth=2, alpha=0.7, zorder=3, label="Running best")

for idx, value in zip(kept_idx, kept_score):
    note = str(valid.loc[idx, "note"]).strip()
    if len(note) > 45:
        note = note[:42] + "..."
    ax.annotate(note, (idx, value),
                textcoords="offset points",
                xytext=(6, 6), fontsize=8.0,
                color="#1a7a3a", alpha=0.9,
                rotation=30, ha="left", va="bottom")

ax.set_xlabel("Experiment #", fontsize=12)
ax.set_ylabel(f"score — {goal} (lower is better)", fontsize=12)
ax.set_title(f"Autoresearch Progress: {len(df)} Experiments, {kept_mask.sum()} Kept "
             f"(goal: {goal})", fontsize=14)
ax.legend(loc="upper right", fontsize=9)
ax.grid(True, alpha=0.2)

best = kept_score.min()
margin = max((baseline - best) * 0.15, 1e-6)
ax.set_ylim(best - margin, baseline + margin)

plt.tight_layout()
plt.savefig("progress.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved to progress.png")

## Summary Statistics

In [ ]:
# Summary stats
kept = df[df["status"] == "keep"].copy()
baseline = df["score"].dropna().iloc[0]
best = kept["score"].min()
best_row = kept.loc[kept["score"].idxmin()]

print(f"Goal:              {df['goal'].iloc[0] if 'goal' in df else 'unknown'}")
print(f"Baseline score:    {baseline:.6f}")
print(f"Best score:        {best:.6f}")
print(f"Total improvement: {baseline - best:.6f} ({(baseline - best) / baseline * 100:.2f}%)")
print(f"Best experiment:   {best_row.get('note', '')}")
if "val_bpb" in df:
    print(f"  its val_bpb:     {best_row['val_bpb']:.6f}")
print()

print("Cumulative effort per improvement:")
for _, row in kept.reset_index().iterrows():
    print(f"  Experiment #{row['index']:3d}: score={row['score']:.6f}  {row.get('note', '')}")

## Top Hits (Kept Experiments by Improvement)

In [ ]:
# Each kept experiment's delta is measured vs the previous kept experiment
# (experiments are cumulative -- each one builds on the last kept state)
kept = df[df["status"] == "keep"].copy()
kept["delta"] = kept["score"].shift(1) - kept["score"]

hits = kept.iloc[1:].sort_values("delta", ascending=False)

print(f"{'Rank':>4}  {'Delta':>9}  {'Score':>10}  Note")
print("-" * 80)
for rank, (_, row) in enumerate(hits.iterrows(), 1):
    print(f"{rank:4d}  {row['delta']:+.6f}  {row['score']:.6f}  {row.get('note', '')}")

print(f"\n{'':>4}  {hits['delta'].sum():+.6f}  {'':>10}  TOTAL improvement over baseline")

## Pareto Frontier

The live ratchet is scalar — it keeps or discards on `score` alone. But every run's
full metric set is in the log, so trade-offs the scalar collapsed can be recovered
after the fact. Change `Y_AXIS` to any numeric column: `peak_vram_gb`,
`num_params_M`, `training_seconds`, or anything a new goal starts recording.

Points on the frontier are the runs not beaten on *both* axes by any other run.
Some will have been discarded by the ratchet.

In [ ]:
X_AXIS = "val_bpb"        # quality
Y_AXIS = "peak_vram_gb"   # cost — swap for num_params_M, training_seconds, ...

pareto_df = df[(df["status"] != "crash")].dropna(subset=[X_AXIS, Y_AXIS]).copy()
pareto_df = pareto_df[np.isfinite(pareto_df[X_AXIS]) & np.isfinite(pareto_df[Y_AXIS])]

if pareto_df.empty:
    print(f"No runs with both {X_AXIS} and {Y_AXIS} recorded yet.")
else:
    # A run is dominated if another is <= on both axes and < on at least one
    xs, ys = pareto_df[X_AXIS].values, pareto_df[Y_AXIS].values
    dominated = np.zeros(len(pareto_df), dtype=bool)
    for i in range(len(pareto_df)):
        better_eq = (xs <= xs[i]) & (ys <= ys[i])
        strictly = (xs < xs[i]) | (ys < ys[i])
        dominated[i] = bool(np.any(better_eq & strictly))
    pareto_df["on_frontier"] = ~dominated

    fig, ax = plt.subplots(figsize=(11, 7))
    off = pareto_df[~pareto_df["on_frontier"]]
    on = pareto_df[pareto_df["on_frontier"]].sort_values(X_AXIS)
    ax.scatter(off[X_AXIS], off[Y_AXIS], c="#cccccc", s=25, label="Dominated", zorder=2)
    ax.scatter(on[X_AXIS], on[Y_AXIS], c="#e74c3c", s=70, zorder=4,
               edgecolors="black", linewidths=0.5, label="Pareto frontier")
    ax.step(on[X_AXIS], on[Y_AXIS], where="post", color="#e74c3c",
            alpha=0.4, linewidth=1.5, zorder=3)
    for _, row in on.iterrows():
        note = str(row.get("note", ""))[:32]
        ax.annotate(note, (row[X_AXIS], row[Y_AXIS]), textcoords="offset points",
                    xytext=(6, 4), fontsize=8, color="#a03227")
    ax.set_xlabel(f"{X_AXIS} (lower is better)")
    ax.set_ylabel(f"{Y_AXIS} (lower is better)")
    ax.set_title(f"{X_AXIS} vs {Y_AXIS} — {len(on)} runs on the frontier of {len(pareto_df)}")
    ax.legend()
    ax.grid(True, alpha=0.2)
    plt.tight_layout()
    plt.show()

    print(f"\nFrontier ({len(on)} runs):")
    for _, row in on.iterrows():
        flag = "" if row["status"] == "keep" else f"  [{row['status']} by the ratchet]"
        print(f"  {X_AXIS}={row[X_AXIS]:.6f}  {Y_AXIS}={row[Y_AXIS]:.2f}  "
              f"{row.get('note', '')}{flag}")